In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder\
        .appName("SkewPractice")\
        .master('local[*]')\
        .config('spark.sql.shuffle.partitions','8')\
        .config('spark.sql.adaptive.enabled','false')\
        .config('spark.sql.adaptive.skewJoin.enabled','false')\
        .config('spark.sql.autoBroadcaseJoinThreshold','-1')\
        .getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 15:41:13 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.190 instead (on interface en0)
26/04/02 15:41:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 15:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark

In [6]:
hot_key_rows = 800000
normal_rows = 200000

hot_df = (
    spark.range(hot_key_rows)
    .withColumn("user_id", lit(101))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

normal_df = (
    spark.range(normal_rows)
    .withColumn("user_id", (col("id") % 5000 + 1000).cast("int"))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

transactions_df = hot_df.union(normal_df)

In [12]:
# users_df = (
#     spark.range(6000)
#     .withColumn("user_id", (col("id") + 1000).cast("int"))
#     .withColumn("segment", expr("CASE WHEN id % 3 = 0 THEN 'gold' WHEN id % 3 = 1 THEN 'silver' ELSE 'bronze' END"))
#     .drop("id")
# )

# hot_user_df = spark.createDataFrame([(101, "platinum")], ["user_id", "segment"])
# hot_user_df.show()
users_df = users_df.union(hot_user_df).dropDuplicates(["user_id"])

In [25]:
baseline_join_df = transactions_df.join(users_df, on="user_id", how="inner")

In [29]:
baseline_join_df.explain()

== Physical Plan ==
*(8) Project [user_id#1, txn_id#2L, amount#3, segment#289]
+- *(8) SortMergeJoin [cast(user_id#1 as bigint)], [user_id#30L], Inner
   :- *(3) Sort [cast(user_id#1 as bigint) ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(cast(user_id#1 as bigint), 8), ENSURE_REQUIREMENTS, [plan_id=1192]
   :     +- Union
   :        :- *(1) Project [101 AS user_id#1, id#0L AS txn_id#2L, cast((rand(-424216914851109763) * 1000.0) as int) AS amount#3]
   :        :  +- *(1) Range (0, 800000, step=1, splits=8)
   :        +- *(2) Filter isnotnull(user_id#5)
   :           +- *(2) Project [cast(((id#4L % 5000) + 1000) as int) AS user_id#5, id#4L AS txn_id#6L, cast((rand(-6842135711656153522) * 1000.0) as int) AS amount#7]
   :              +- *(2) Range (0, 200000, step=1, splits=8)
   +- SortAggregate(key=[user_id#30L], functions=[first(segment#13, false)])
      +- *(7) Sort [user_id#30L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(user_id#30L, 8), EN

In [30]:
baseline_join_df.count()

1000000

In [31]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64MB")

In [33]:
aqe_join_df = transactions_df.join(users_df, on="user_id", how="inner")
aqe_join_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#1, txn_id#2L, amount#3, segment#313]
   +- SortMergeJoin [cast(user_id#1 as bigint)], [user_id#30L], Inner
      :- Sort [cast(user_id#1 as bigint) ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(cast(user_id#1 as bigint), 8), ENSURE_REQUIREMENTS, [plan_id=1524]
      :     +- Union
      :        :- Project [101 AS user_id#1, id#0L AS txn_id#2L, cast((rand(-424216914851109763) * 1000.0) as int) AS amount#3]
      :        :  +- Range (0, 800000, step=1, splits=8)
      :        +- Filter isnotnull(user_id#5)
      :           +- Project [cast(((id#4L % 5000) + 1000) as int) AS user_id#5, id#4L AS txn_id#6L, cast((rand(-6842135711656153522) * 1000.0) as int) AS amount#7]
      :              +- Range (0, 200000, step=1, splits=8)
      +- SortAggregate(key=[user_id#30L], functions=[first(segment#13, false)])
         +- Sort [user_id#30L ASC NULLS FIRST], false, 0
            +- Exchange hashpa

In [34]:
aqe_join_df.count()

1000000